In [8]:
import numpy as np
import torch
import tqdm
import itertools
import shutil

from pathlib import Path
from dataclasses import dataclass

from typing import Tuple, List, Dict, Optional

from src.episodes.history import IntrinsicHistory, History
from src.controller.mfld_plant_dyn import ManualManifoldPlantDynamics
from src.manifolds.sn_mfld import HypersphereManifold

In [9]:
UNFORCED_DYNAMICS_DATA_DIR = Path("../../../data/unforced_dynamics")
FORCED_DYNAMICS_DATA_DIR = Path("../data/forced_dynamics_data")

DATA_SUBFOLDER_FORMAT = "dim_{dim}/radius_{radius}"
DATA_FILE_FORMAT = "ic_{ic_idx}.npz"

UNIT_RADIUS_VEL_COORD_STD = 0.2

MIN_HS_DIM = 3
MAX_HS_DIM = 3
NUM_ICS_PER_DIM = 400

RADII = [1.0, 2.0, 0.5]

# each trajectory is 60 seconds
STEP_DT = 0.05  # 20 Hz
NUM_STEPS = 300

NONZERO_EPS = 1e-6


In [10]:
def project_pos_onto_hs(radius: float, pos: np.ndarray) -> np.ndarray:
    return radius * pos / np.linalg.norm(pos)


def project_vel_onto_hs(radius: float, pos: np.ndarray, vel: np.ndarray) -> np.ndarray:
    return radius * (vel / np.linalg.norm(pos) - np.dot(pos, vel) / np.linalg.norm(pos) ** 3 * pos)


def generate_extrinsic_ic_on_hs(n: int, radius: float, vel_coord_std: float, rand: np.random.Generator) -> Tuple[
    np.ndarray, np.ndarray]:
    # choose a random position in the ambient space which we will project onto the surface of the hypersphere
    # NOTE: projection onto the hypersphere will fail if the norm of the position is 0

    ambient_n = n + 1

    pos_ambient = np.zeros((ambient_n,))
    while np.linalg.norm(pos_ambient) < NONZERO_EPS:
        pos_ambient = rand.multivariate_normal(np.zeros((ambient_n,)), radius ** 2.0 * np.eye(ambient_n))
    pos_on_hs = project_pos_onto_hs(radius, pos_ambient)

    vel_ambient = rand.multivariate_normal(np.zeros((ambient_n,)), vel_coord_std ** 2.0 * np.eye(ambient_n))
    vel_on_hs = project_vel_onto_hs(radius, pos_ambient, vel_ambient)

    return pos_on_hs, vel_on_hs

In [11]:
def generate_history(n: int, radius: float, pos_initial: np.ndarray, vel_initial: np.ndarray, dt: float,
                     num_steps: int, rand: np.random.Generator,
                     controls_coord_std: Optional[float] = None) -> History:
    ambient_n = n + 1

    hs_manifold = HypersphereManifold(n, radius)

    # transforms intrinsic representation (in ambient euclidean space) into an intrinsic representation in coordinate chart far from the singularities of the parameterization
    ns_chart = hs_manifold.nonsingular_chart_id(torch.tensor(pos_initial))
    pos_initial_ns_intrinsic = hs_manifold.to_intrinsic(ns_chart, torch.tensor(pos_initial)).detach().numpy()
    vel_initial_ns_intrinsic = hs_manifold.to_intrinsic_ts(ns_chart, torch.tensor(pos_initial),
                                                           torch.tensor(vel_initial)).detach().numpy()

    hs_dynamics = ManualManifoldPlantDynamics(hs_manifold,
                                              (ns_chart, pos_initial_ns_intrinsic, vel_initial_ns_intrinsic), )

    # sets up the multivariate distribution for the controls (note that if not provided then controls will be identically zero)
    controls_mean = np.zeros((ambient_n,))
    controls_covar = controls_coord_std ** 2.0 * np.eye(ambient_n) if controls_coord_std is not None else np.zeros(
        (ambient_n, ambient_n))

    generated_history = History(
        sample_time=dt,
        uses_controls=controls_coord_std is not None,

        extrinsic_pos=np.zeros((num_steps, ambient_n)),
        extrinsic_vel=np.zeros((num_steps, ambient_n)),
        extrinsic_u=np.zeros((num_steps, ambient_n)),

        intrinsic={
            chart: IntrinsicHistory(
                pos=np.zeros((num_steps, n)),
                vel=np.zeros((num_steps, n)),
                u=np.zeros((num_steps, n)),
                valid=np.zeros((num_steps, n)),
            ) for chart in hs_manifold.charts
        }
    )

    for timestep_idx in range(num_steps):
        current_pos_extrinsic, current_vel_extrinsic = hs_dynamics.current_state_extrinsic
        current_controls_extrinsic = project_vel_onto_hs(radius, current_pos_extrinsic,
                                                         rand.multivariate_normal(controls_mean, controls_covar))

        generated_history.extrinsic_pos[timestep_idx, :] = current_pos_extrinsic
        generated_history.extrinsic_vel[timestep_idx, :] = current_vel_extrinsic
        generated_history.extrinsic_u[timestep_idx, :] = current_controls_extrinsic

        for chart in hs_manifold.charts:
            # NOTE: the basis check must be ignored here as given we're generating this data for all charts then it is inevitable
            # that we will encounter positions on the manifolds that are at coordinate chart singularities for some of the charts
            # so the tangent space will be rank-deficient (lossy projection of extrinsic velocity into intrinsic coordinates)

            current_pos_intrinsic = hs_manifold.to_intrinsic(chart,
                                                             torch.tensor(current_pos_extrinsic)).detach().numpy()
            current_vel_intrinsic = hs_manifold.to_intrinsic_ts(chart,
                                                                torch.tensor(current_pos_extrinsic),
                                                                torch.tensor(current_vel_extrinsic),
                                                                ignore_basis_check=True).detach().numpy()
            current_controls_intrinsic = hs_manifold.to_intrinsic_ts(chart,
                                                                     torch.tensor(current_pos_extrinsic),
                                                                     torch.tensor(current_controls_extrinsic),
                                                                     ignore_basis_check=True).detach().numpy()

            current_coords_validity = hs_manifold.intrinsic_coords_validity(chart, torch.tensor(
                current_pos_intrinsic), ).detach().numpy()

            generated_history.intrinsic[chart].pos[timestep_idx, :] = current_pos_intrinsic
            generated_history.intrinsic[chart].vel[timestep_idx, :] = current_vel_intrinsic
            generated_history.intrinsic[chart].u[timestep_idx, :] = current_controls_intrinsic
            generated_history.intrinsic[chart].valid[timestep_idx, :] = current_coords_validity

        hs_dynamics.step(dt, inputs_extrinsic=current_controls_extrinsic)

    return generated_history

In [12]:
# change these to adjust whether unforced/forced dynamics are used
DYNAMICS_DATA_DIR = UNFORCED_DYNAMICS_DATA_DIR
CONTROLS_COORD_STD = None

In [13]:
# # clear the pre-existing data inside the unforced folder
# for child in DYNAMICS_DATA_DIR.iterdir():
#     if child.is_dir():
#         shutil.rmtree(child.resolve())
#     elif child.is_file() and child.name != ".gitkeep":
#         child.unlink()
#
# # re-generate the necessary subfolders
# for dim, radius in itertools.product(range(MIN_HS_DIM, MAX_HS_DIM + 1), RADII):
#     subdir_folder = str.format(DATA_SUBFOLDER_FORMAT, dim=dim, radius=radius)
#     subdir_folder.replace(".", "_")  # due to the decimal point
#
#     subdir_path = DYNAMICS_DATA_DIR / subdir_folder
#     subdir_path.mkdir(parents=True, exist_ok=True)

In [14]:
rand = np.random.default_rng(42)

cfgs = list(itertools.product(range(MIN_HS_DIM, MAX_HS_DIM + 1), RADII, range(NUM_ICS_PER_DIM//4)))
for dim, radius, ic_idx in tqdm.tqdm(cfgs, desc="Runs", total=len(cfgs)):
    # generates an initial history and runs a simulation over the whole duration
    initial_pos_extrinsic, initial_vel_extrinsic = generate_extrinsic_ic_on_hs(dim,
                                                                               radius,
                                                                               radius * UNIT_RADIUS_VEL_COORD_STD,
                                                                               rand)
    history = generate_history(dim, radius, initial_pos_extrinsic, initial_vel_extrinsic, STEP_DT, NUM_STEPS, rand,
                               CONTROLS_COORD_STD)

    # generates the save file path
    subdir_folder = str.format(DATA_SUBFOLDER_FORMAT, dim=dim, radius=radius)
    subdir_folder.replace(".", "_")  # due to the decimal point

    filename = str.format(DATA_FILE_FORMAT, ic_idx=ic_idx)

    path = DYNAMICS_DATA_DIR / subdir_folder / filename
    history.save(path)

Runs:   0%|          | 0/300 [00:10<?, ?it/s]


KeyboardInterrupt: 